# ADVS — YOLOv8 Signature + Stamp/Seal + Logo Detector

Model 2 of 4 — detects three classes (0=signature, 1=stamp_seal, 2=logo) on full document pages (`training_script.md` §2). Trains `yolov8n.pt` and exports `yolov8_signature_stamp_logo.onnx`.

> Self-contained notebook. The code cell below is the full source of `scripts/train_detector.py` (definitions only); the run cell at the bottom launches training. Edit `DATA_ROOT` to point at your data.

## 1. Install dependencies (Colab / fresh env)

Skip if your environment already has the ADVS ML stack installed.

In [ ]:
# !pip install tensorflow==2.16.* ultralytics scikit-learn opencv-python-headless onnx onnxruntime pandas pillow

## 1b. Mount Google Drive (Colab only)

Skip on a local Jupyter server. On Colab, your dataset (`training/detector_data/{images,labels}`,
`validation/detector_data/{images,labels}`) and `models/` output almost never live in the ephemeral
runtime filesystem — mount Drive and point `DATA_ROOT`/`MODELS_OUT` (config cell below) at a path
under `/content/drive/MyDrive/...`.

In [ ]:
"""ADVS - YOLOv8 signature + stamp/seal + logo detector (model 2 of 4).

Detects three classes (0=signature, 1=stamp_seal, 2=logo) on full document pages. Faithful to
training_script.md §2.

Data layout (read-only):
    <data-root>/training/detector_data/images/*.jpg|png
    <data-root>/training/detector_data/labels/*.txt   (YOLO: "<cls> cx cy w h")
    <data-root>/validation/detector_data/images/*.jpg|png
    <data-root>/validation/detector_data/labels/*.txt

Outputs (under <models-out>):
    detector_data.yaml (generated), yolov8_signature_stamp_logo.onnx, yolo_runs/

Run:
    python scripts/train_detector.py                 # full training (needs ultralytics + data)
    python scripts/train_detector.py --dry-run       # validate layout only (stdlib only)
    python scripts/train_detector.py --smoke         # 1-epoch tiny CPU run (needs ultralytics)

Heavy imports (ultralytics, torch) are lazy so --dry-run works with only stdlib.
"""

from __future__ import annotations

import argparse
import sys
import time
from pathlib import Path

try:
    PY_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    PY_ROOT = Path.cwd()  # notebook fallback — assumes cwd is python/

CONFIG: dict = {
    "weights": "yolov8n.pt",
    "epochs": 50,
    "imgsz": 640,
    "batch": 16,
    "patience": 10,
}
SMOKE_OVERRIDES = {
    "epochs": 1,
    "imgsz": 64,
    "batch": 2,
    "patience": 2,
}
NAMES = {0: "signature", 1: "stamp_seal", 2: "logo"}
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}


def log(msg: str) -> None:
    print(f"[detector] {msg}", flush=True)


def section(title: str) -> None:
    print("\n" + "=" * 70 + f"\n  {title}\n" + "=" * 70, flush=True)


class TrainError(RuntimeError):
    """Expected, user-actionable failure (e.g. missing data)."""


def require_dir(path: Path, what: str) -> None:
    if not path.is_dir():
        raise TrainError(f"Missing {what}: expected directory '{path}'.")


def count_images(path: Path) -> int:
    return sum(1 for p in path.rglob("*") if p.suffix.lower() in IMG_EXTS)


def validate_structure(train_dir: Path, val_dir: Path) -> dict:
    require_dir(train_dir / "images", "detector training images")
    require_dir(train_dir / "labels", "detector training labels")
    require_dir(val_dir / "images", "detector validation images")
    n_train = count_images(train_dir / "images")
    n_val = count_images(val_dir / "images")
    if n_train == 0:
        raise TrainError(f"No training images found in {train_dir / 'images'}")
    return {"train_images": n_train, "val_images": n_val}


def write_data_yaml(yaml_path: Path, data_root: Path) -> None:
    """Ultralytics resolves label paths by swapping 'images'->'labels'."""
    text = (
        f"path: {data_root.as_posix()}\n"
        f"train: training/detector_data/images\n"
        f"val: validation/detector_data/images\n"
        f"names:\n" + "".join(f"  {k}: {v}\n" for k, v in NAMES.items())
    )
    yaml_path.write_text(text)
    log(f"Wrote {yaml_path}")


def pick_device():
    try:
        import torch

        return 0 if torch.cuda.is_available() else "cpu"
    except Exception:  # noqa: BLE001
        return "cpu"


def train(cfg: dict, data_root: Path, models_out: Path) -> None:
    section("YOLOv8 signature + stamp/seal + logo detection")
    from ultralytics import YOLO

    yaml_path = models_out / "detector_data.yaml"
    write_data_yaml(yaml_path, data_root)

    device = pick_device()
    log(f"Training {cfg['weights']} for {cfg['epochs']} epochs "
        f"(imgsz={cfg['imgsz']}, batch={cfg['batch']}, device={device})")
    model = YOLO(cfg["weights"])
    model.train(
        data=str(yaml_path), epochs=cfg["epochs"], imgsz=cfg["imgsz"],
        batch=cfg["batch"], patience=cfg["patience"], device=device, cache=True,
        project=str(models_out / "yolo_runs"), name="train", exist_ok=True,
    )

    metrics = model.val()
    try:
        log(f"mAP@0.5 = {float(metrics.box.map50):.4f}")
    except Exception:  # noqa: BLE001 - metrics shape varies by version
        log("validation complete (mAP attribute unavailable on this version)")

    onnx_path = models_out / "yolov8_signature_stamp_logo.onnx"
    exported = model.export(format="onnx", imgsz=cfg["imgsz"])
    import shutil

    if exported and Path(exported).exists() and Path(exported) != onnx_path:
        shutil.copy(str(exported), str(onnx_path))
    log(f"Exported {onnx_path}")
    if onnx_path.exists():
        log("inference sanity check -> yolov8_signature_stamp_logo.onnx present")


def parse_args(argv: list[str]) -> argparse.Namespace:
    ap = argparse.ArgumentParser(description="Train the ADVS YOLOv8 detector.")
    ap.add_argument("--data-root", default=str(PY_ROOT / "data"))
    ap.add_argument("--models-out", default=str(PY_ROOT / "models"))
    ap.add_argument("--dry-run", action="store_true",
                    help="Validate layout/config only; no heavy imports, no training.")
    ap.add_argument("--smoke", action="store_true",
                    help="Tiny 1-epoch CPU run (needs ultralytics + fixtures).")
    return ap.parse_args(argv)


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv if argv is not None else sys.argv[1:])
    cfg = dict(CONFIG)
    if args.smoke:
        cfg.update(SMOKE_OVERRIDES)
        log("SMOKE MODE: reduced epochs/sizes, CPU only.")

    data_root = Path(args.data_root)
    train_dir = data_root / "training" / "detector_data"
    val_dir = data_root / "validation" / "detector_data"
    models_out = Path(args.models_out)
    log(f"data-root={data_root}  models-out={models_out}")

    try:
        summary = validate_structure(train_dir, val_dir)
    except TrainError as exc:
        log(f"STRUCTURE ERROR: {exc}")
        return 2
    log(f"ok: {summary}")

    if args.dry_run:
        section("DRY RUN - structure valid, skipping training")
        return 0

    models_out.mkdir(parents=True, exist_ok=True)
    started = time.time()
    train(cfg, data_root, models_out)
    section(f"Done in {time.time() - started:.1f}s - artefacts in {models_out}")
    return 0

## 2. Training code (from `scripts/train_detector.py`)

Running this cell only defines functions — it does not start training.

In [ ]:
from pathlib import Path

if "IN_COLAB" in dir() and IN_COLAB:
    DATA_ROOT = "/content/drive/MyDrive/advs/data"      # edit to your Drive dataset location
    MODELS_OUT = "/content/drive/MyDrive/advs/models"   # edit to your Drive output location
else:
    DATA_ROOT = str(PY_ROOT / "data")     # edit to your dataset location
    MODELS_OUT = str(PY_ROOT / "models")
print("data:", DATA_ROOT, "\nout :", MODELS_OUT)

## 3. Configure paths

`DATA_ROOT` must contain `training/` and `validation/` subfolders for this model (see `python/data/README.md`).

In [ ]:
from pathlib import Path
DATA_ROOT = str(PY_ROOT / "data")     # edit to your dataset location
MODELS_OUT = str(PY_ROOT / "models")
print("data:", DATA_ROOT, "
out :", MODELS_OUT)

## 4. Validate the data layout (no training)

A quick structural check before committing to a full run.

In [ ]:
main(["--data-root", DATA_ROOT, "--models-out", MODELS_OUT, "--dry-run"])

## 5. Train

Use `--smoke` for a fast 1-epoch CPU sanity run, or drop it for the full training schedule (needs real data + ideally a GPU).

In [ ]:
main(["--data-root", DATA_ROOT, "--models-out", MODELS_OUT, "--smoke"])
# Full run: main(["--data-root", DATA_ROOT, "--models-out", MODELS_OUT])